# CNNs and Ice core data

In this exercise we are going to be classifying ashes found in ice core drilling. We will be dealing with a dataset (provided by Niccolo Maffezzoli, a former TA in 'Applied Statistics') that labels images and meta-data into two types of volcanic ashes: _Grimsvotn_ og _Campanian_.

The data for this exercise consists of both scalar values (.csv files) and images (in directories), already split into training and testing samples, and can be found here:
https://sid.erda.dk/share_redirect/gqwa15no19

Your task will be to make the best classifier. First you will make a classifier based on the image data alone, then one based on the meta data alone, and finally one that combines both. You are free to use whatever methods you wish, but ideas will be provided... :)

NOTE: The data should be placed *in the same directory as the code* in order for the code to run out of the box.

***

Author: Niccolo Maffezzoli, Amalie Mygind, and Troels Petersen

Email: petersen@nbi.dk

Date: 1st of May 2025 (latest version)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from PIL import Image
from tqdm import tqdm

### Loading, splitting the data

In [ ]:
data_dir = './datasets/icecore/'

train_df = pd.read_csv(data_dir + 'supervised_train.csv')
train_df.imgpaths = data_dir + train_df.imgpaths.astype(str)

# Make sure to stratify if label imbalance is suspected
train, val = train_test_split(train_df, test_size=0.3, random_state=42, stratify=train_df['class'])

test = pd.read_csv(data_dir + 'supervised_test.csv')
test.imgpaths = data_dir + test.imgpaths.astype(str)


In [ ]:
train_df.describe()
train_df.columns

We will be using PyTorch's DataLoader to make batches of our data.

In [ ]:
cols_mva = ['Area (ABD)', 'Area (Filled)', 'Aspect Ratio', 'Biovolume (Cylinder)',
       'Biovolume (P. Spheroid)', 'Circle Fit',
       'Circularity', 'Circularity (Hu)', 'Compactness', 'Convex Perimeter',
       'Convexity', 'Diameter (ABD)', 'Diameter (ESD)', 'Edge Gradient',
       'Elongation', 'Feret Angle Max', 'Feret Angle Min', 'Fiber Curl',
       'Fiber Straightness', 'Geodesic Aspect Ratio', 'Geodesic Length',
       'Geodesic Thickness', 'Intensity', 'Length', 'Particles Per Chain',
       'Perimeter', 'Roughness', 'Sigma Intensity', 'Sum Intensity',
       'Symmetry', 'Transparency', 'Volume (ABD)', 'Volume (ESD)', 'Width']


# Preprocess meta-feature matrices once and properly (avoid leakage)
scaler = StandardScaler()
X_train_features = scaler.fit_transform(train[cols_mva])
X_val_features = scaler.transform(val[cols_mva])
X_test_features = scaler.transform(test[cols_mva])

# Dataset class
class ParticleDataset(torch.utils.data.Dataset):
    def __init__(self, df, features, transform=None):
        self.df = df.reset_index(drop=True)
        self.imgpaths = df['imgpaths'].to_numpy()
        self.labels = df['class'].to_numpy()
        self.X_features = features
        self.transform = transform or transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[94 / 255], std=[12 / 255]),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Load grayscale image
        img = Image.open(self.imgpaths[idx]).convert('L')
        image = self.transform(img)

        label = torch.tensor(self.labels[idx]).long()
        xfeatures = torch.tensor(self.X_features[idx]).float()

        return image, label, xfeatures

# Define transforms
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[94 / 255], std=[12 / 255]),
])

eval_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[94 / 255], std=[12 / 255]),
])

# Instantiate datasets
train_dataset = ParticleDataset(train, X_train_features, transform=train_transform)
val_dataset = ParticleDataset(val, X_val_features, transform=eval_transform)
test_dataset = ParticleDataset(test, X_test_features, transform=eval_transform)

# DataLoaders
batch_size = 32

dataloader_train = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
dataloader_val = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
dataloader_test = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)


In [ ]:
# cols_mva = ['Area (ABD)', 'Area (Filled)', 'Aspect Ratio', 'Biovolume (Cylinder)',
#        'Biovolume (P. Spheroid)', 'Circle Fit',
#        'Circularity', 'Circularity (Hu)', 'Compactness', 'Convex Perimeter',
#        'Convexity', 'Diameter (ABD)', 'Diameter (ESD)', 'Edge Gradient',
#        'Elongation', 'Feret Angle Max', 'Feret Angle Min', 'Fiber Curl',
#        'Fiber Straightness', 'Geodesic Aspect Ratio', 'Geodesic Length',
#        'Geodesic Thickness', 'Intensity', 'Length', 'Particles Per Chain',
#        'Perimeter', 'Roughness', 'Sigma Intensity', 'Sum Intensity',
#        'Symmetry', 'Transparency', 'Volume (ABD)', 'Volume (ESD)', 'Width']

# class ParticleDataset():
#     # This class is simply a "list-like" object
#     # that will load an image on the fly (using Image.open).
    
#     def __init__(self, df):
#         self.df = df
#         self.imgpaths = df['imgpaths'].to_numpy()
#         # self.imgpaths = df[path+'train/'].to_numpy()
#         self.labels = df['class'].to_numpy()
#         scaler = StandardScaler()
#         self.X_features = scaler.fit_transform(df[cols_mva])
        
#         # Feel free to change these transforms! They are not optimal.
#         # You could also include e.g. RandomHorizontalFlip, RandomVerticalFlip
#         # for data augmentation.
#         self.transform = transforms.Compose([
#             transforms.Resize((128, 128)), 
#             transforms.ToTensor(),
#             transforms.Normalize(mean=[94/255], std=[12/255]),
#             ])

#     def __len__(self):
#         return len(self.df)   # you can limit the number of images by returning a smaller number here

#     def __getitem__(self, idx):
#         imgpath = self.imgpaths[idx]
#         image = self.transform(Image.open(imgpath))        
        
#         label = torch.tensor(self.labels[idx]).int()
#         xfeatures = torch.from_numpy(self.X_features[idx]).float()
        
#         return image, label, xfeatures

    
# batch_size = 32

# dataloader_train = DataLoader(ParticleDataset(train), batch_size=batch_size, shuffle=True)
# dataloader_val = DataLoader(ParticleDataset(val), batch_size=batch_size, shuffle=True)
# dataloader_test = DataLoader(ParticleDataset(test), batch_size=batch_size, shuffle=True)

We now have data loaders that can sample from our data. These can be used in a loop:

In [ ]:
img_batch, label_batch, meta_batch = next(iter(dataloader_train))
print('Image batch shape:', img_batch.shape)   # [B, C, H, W]
print('Label batch shape:', label_batch.shape)
print('Meta features shape:', meta_batch.shape)


In [ ]:
# for img, label, meta_features in dataloader_train:
#     print('Image batch shape =', img.shape)
#     print('Label batch shape =', label.shape)
#     print('Meta features batch shape =', meta_features.shape)
#     break

In [ ]:
all_labels = torch.cat([labels for _, labels, _ in dataloader_train])
class_counts = all_labels.bincount()
plt.bar(range(len(class_counts)), class_counts.numpy())
plt.xticks(range(len(class_counts)))
plt.xlabel("Class")
plt.ylabel("Frequency")
plt.title("Label distribution in training set")
plt.show()


In [ ]:
# labels = []

# for _, label, _ in dataloader_train:
#     labels.append(label)

# labels = torch.cat(labels)
# # histogram of labels
# plt.hist(labels.numpy(), bins=range(3))

Let us have a look at a few of the images:

In [ ]:
plt.figure(figsize=(7, 7))
img_batch, label_batch, _ = next(iter(dataloader_train))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(img_batch[i, 0].numpy(), cmap='gray')
    plt.axis('off')
    plt.title(f'Label = {label_batch[i].item()}')
plt.tight_layout()
plt.show()


In [ ]:
# # Try running this cell many times -- new batches are generated each time:
# plt.figure(figsize=(7, 7))
# for img, label, meta_features in dataloader_train:
#     for i in range(9):
#         plt.subplot(3, 3, i + 1)
#         plt.imshow(img[i, 0, :, :])
#         plt.axis('off')
#         plt.title(f'Label = {int(label[i])}')
#     break

## An example network:

This network is rather small and does not include many of the features it potentially could (see exercise 1).

Once again, try to draw the CNN, and think about how many parameters it has.

In [ ]:
# # Input image size = 1 x 128 x 128  (1 channel, i.e. bw)
# net = nn.Sequential(nn.Conv2d(1, 3, 5),  # shape = (3, 124 x 124)
#                     nn.ReLU(),           # shape = (3, 124 x 124), see https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
#                     nn.MaxPool2d(10),    # shape = (3, 12 x 12), see https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html
#                     nn.Flatten(),        # shape = (3 * 12 * 12) = (432)
#                     nn.Linear(432, 1),   # shape = (1)
#                     nn.Sigmoid())        # shape (1)

This network takes a batch of images (batch x 1 x image_width x image_height) and returns a single number for each image (batch x 1):

In [ ]:
# for img, label, meta_features in dataloader_train:
#     print(net(img).shape)
#     break

You do not need to calculate shapes yourself. You can always ask with `.shape`.

We could also have made this networks more explicitely by using a class:

In [ ]:
# class IceCoreNet(nn.Module):
#     def __init__(self, n_meta_features, n_classes):
#         super(IceCoreNet, self).__init__()

#         # Convolutional image encoder
#         self.conv = nn.Sequential(
#             nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 1x128x128 → 32x128x128
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.MaxPool2d(2),  # → 32x64x64

#             nn.Conv2d(32, 64, kernel_size=3, padding=1),  # → 64x64x64
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.MaxPool2d(2),  # → 64x32x32

#             nn.Conv2d(64, 128, kernel_size=3, padding=1),  # → 128x32x32
#             nn.BatchNorm2d(128),
#             nn.ReLU(),
#             nn.AdaptiveAvgPool2d((1, 1))  # → 128x1x1
#         )

#         # Tabular feature encoder
#         self.meta = nn.Sequential(
#             nn.Linear(n_meta_features, 64),
#             nn.ReLU(),
#             nn.BatchNorm1d(64),
#             nn.Dropout(0.2)
#         )

#         # Final classification head
#         self.classifier = nn.Sequential(
#             nn.Linear(128 + 64, 128),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(128, n_classes)
#         )

#     def forward(self, image, meta):
#         x_img = self.conv(image)
#         x_img = x_img.view(x_img.size(0), -1)  # Flatten to (B, 128)
        
#         x_meta = self.meta(meta)

#         x = torch.cat([x_img, x_meta], dim=1)
#         return self.classifier(x)

class IceCoreNet(nn.Module):
    def __init__(self, n_meta_features):
        super(IceCoreNet, self).__init__()

        # Convolutional branch for images
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # [B, 32, 128, 128]
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                             # [B, 32, 64, 64]

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # [B, 64, 64, 64]
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                             # [B, 64, 32, 32]

            nn.Conv2d(64, 128, kernel_size=3, padding=1),# [B, 128, 32, 32]
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))                 # [B, 128, 1, 1]
        )

        # Metadata branch
        self.meta = nn.Sequential(
            nn.Linear(n_meta_features, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2)
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(128 + 64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),        # Binary output
            # nn.Sigmoid()              # Probabilities in [0, 1]
        )

    # def forward(self, image, meta):
    #     x_img = self.conv(image).view(image.size(0), -1)  # flatten
    #     x_meta = self.meta(meta)
    #     x = torch.cat([x_img, x_meta], dim=1)
    #     return self.classifier(x)
    # def forward(self, image, meta):
    #     # Handle image input: if image is None, create a tensor of zeros (or another placeholder)
    #     if image is not None:
    #         x_img = self.conv(image).view(image.size(0), -1)  # Flatten the output
    #     else:
    #         # Create a tensor of zeros with the appropriate size (assuming B = batch size, 128 = output channels from conv layer)
    #         x_img = torch.zeros(image.size(0), 128, device=image.device)  # Adjust shape accordingly

    #     # Handle meta input: if meta is None, create a tensor of zeros
    #     if meta is not None:
    #         x_meta = self.meta(meta)
    #     else:
    #         # Create a tensor of zeros for metadata features
    #         x_meta = torch.zeros(image.size(0), 64, device=meta.device)  # Adjust shape accordingly

    #     # Concatenate image and meta features
    #     x = torch.cat([x_img, x_meta], dim=1)
        
    #     return self.classifier(x)
    
    def forward(self, image, meta):
        # Handle image input: if image is None, create a tensor of zeros (or another placeholder)
        if image is not None:
            x_img = self.conv(image).view(image.size(0), -1)  # Flatten the output
        else:
            # Create a tensor of zeros with the appropriate size (assuming B = batch size, 128 = output channels from conv layer)
            x_img = torch.zeros(image.size(0), 128, device=image.device)  # Adjust shape accordingly

        # Handle meta input: if meta is None, create a tensor of zeros
        if meta is not None:
            x_meta = self.meta(meta)
        else:
            # If meta is None, create a tensor of zeros (with the correct shape) without accessing meta.device
            x_meta = torch.zeros(image.size(0), 64, device=image.device)  # Adjust shape accordingly

        # Concatenate image and meta features
        x = torch.cat([x_img, x_meta], dim=1)
        
        return self.classifier(x)




model = IceCoreNet(n_meta_features=len(cols_mva))

# Move model to appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


In [ ]:
# class Net(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.conv1 = nn.Conv2d(1, 3, 5)
#         self.nonlinearity = nn.ReLU()
#         self.mp1 = nn.MaxPool2d(10)
#         self.linear1 = nn.Linear(432, 1)
#         self.to_prop = nn.Sigmoid()
        
#     def forward(self, x):
#         x = self.conv1(x)
#         x = self.nonlinearity(x)
#         x = self.mp1(x)
#         return self.to_prop(self.linear1(x.view(x.shape[0], -1)))
    
# net = Net()

In [ ]:
# # Fetch one batch from the dataloader
# for img, label, meta_features in dataloader_train:
#     output = model(img, meta_features)  # ✅ Pass both image and meta
#     print(output.shape)
#     break

# Set model to eval mode if testing (avoids dropout, uses batchnorm running stats)
model.eval()

# Fetch a batch
for img, label, meta_features in dataloader_train:
    output = model(img, meta_features)  # (batch_size, 3)
    print("Output logits shape:", output.shape)
    print("Sample logits:", output[0])  # Logits for one sample
    break

In [ ]:
# for img, label, meta_features in dataloader_train:
#     print(net(img).shape)
#     break

The two definitions are exactly the same, but, naturally, the latter is a lot more customizable.

We are going to intepret the output of the model as being the probability of class 1. The `Sigmoid` at the end ensures that it is a number in [0, 1]

In [ ]:
for img, label, meta_features in dataloader_train:
    output = model(img, meta_features)  # (batch_size, 3)
    print(output.min(), output.max())
    break

In [ ]:
# for img, label, meta_features in dataloader_train:
#     output = net(img)
#     print(output.min(), output.max())
#     break

## An example training routine

All the parameters of our model can be found using `.parameters()`. We make an optimizer and tell it to optimize over these paramters:

In [ ]:
# Define optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

In [ ]:
# Optionally add learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

In [ ]:
# opt = torch.optim.Adam(net.parameters(), lr=1e-4)  # lr = learning rate

We are also going to use a GPU, if we have one:

In [ ]:
# Move model to device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print('Running on', device)

In [ ]:
# # device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = "cpu"
# net.to(device)
# print('Running on', device)

To train we loop over epochs and the dataset:

In [ ]:
# Storage for logging
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

num_epochs = 10  # adjust as needed

def evaluate(model, dataloader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for img, label, meta_features in dataloader:
            img = img.to(device)
            meta_features = meta_features.to(device)
            label = label.float().to(device)

            output = model(img, meta_features).squeeze()
            loss = criterion(output, label)
            running_loss += float(loss)

            preds = (torch.sigmoid(output) > 0.5).float()
            correct += (preds == label).sum().item()
            total += len(label)

    avg_loss = running_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

# Training loop

criterion = nn.BCEWithLogitsLoss()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for img, label, meta_features in tqdm(dataloader_train, desc=f"Epoch {epoch+1}/{num_epochs}"):
        img = img.to(device)
        meta_features = meta_features.to(device)
        label = label.float().to(device)

        output = model(img, meta_features).squeeze()
        loss = criterion(output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += float(loss)
        preds = (torch.sigmoid(output) > 0.5).float()
        correct += (preds == label).sum().item()
        total += len(label)

    # Training metrics
    train_loss = running_loss / len(dataloader_train)
    train_acc = correct / total

    # Validation metrics
    val_loss, val_acc = evaluate(model, dataloader_val, criterion)

    # Learning rate scheduling
    scheduler.step(val_loss)

    # Logging
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Train Acc = {train_acc:.4f} | Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.4f}")

# Plotting
epochs = range(1, num_epochs + 1)

plt.figure(figsize=(12, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, label='Train Loss')
plt.plot(epochs, val_losses, label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss per Epoch")
plt.legend()

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(epochs, train_accuracies, label='Train Accuracy')
plt.plot(epochs, val_accuracies, label='Validation Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy per Epoch")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# loss_criterion = nn.BCELoss()
# for epoch in range(5):
#     acc_loss = 0.0
#     for img, label, meta_features in tqdm(dataloader_train, desc=f'Epoch {epoch + 1}'):
#         img = img.to(device)
#         output = net(img).squeeze()
#         label = label.float().to(device)
#         loss = loss_criterion(output, label)
#         loss.backward()  # calculate gradients: d Loss / d Paramters
#         opt.step()  # take a step down-hill
#         opt.zero_grad()  # zero the gradient calculations for next iteration
#         acc_loss += float(loss)
#     acc_loss /= len(dataloader_train)
#     print(f'Epoch {epoch + 1} loss = {acc_loss}')
    

After we have trained a model, we should evaluate its accuracy:

In [ ]:
# def calc_accuracy(model, dataset, device, only_img=True, only_meta=False):
#     acc = 0.0
#     count = 0
#     model.eval()
#     with torch.no_grad():
#         for img, label, meta_features in dataset:
#             label = label.to(device)
#             img = img.to(device)
#             meta_features = meta_features.to(device)

#             if only_img:
#                 output = model(img, None).squeeze()
#             elif only_meta:
#                 output = model(None, meta_features).squeeze()
#             else:
#                 output = model(img, meta_features).squeeze()

#             prediction = torch.round(output)
#             acc += torch.sum(prediction == label)
#             count += len(label)
#     return acc / count

def calc_accuracy(model, dataset, device, only_img=True, only_meta=False):
    acc = 0.0
    count = 0
    model.eval()
    with torch.no_grad():
        for img, label, meta_features in dataset:
            label = label.to(device)
            img = img.to(device)
            meta_features = meta_features.to(device)

            if only_img:
                output = model(img, None).squeeze()
            elif only_meta:
                output = model(None, meta_features).squeeze()
            else:
                output = model(img, meta_features).squeeze()

            prediction = torch.round(output)
            acc += torch.sum(prediction == label)
            count += len(label)
    return acc / count



# Accuracy calculation
print(f'Train accuracy = {float(calc_accuracy(model, dataloader_train, device)):.4f}')
print(f'Test accuracy  = {float(calc_accuracy(model, dataloader_test, device)):.4f}')


In [ ]:
# def calc_accuracy(model, dataset, only_img=True, only_meta=False):
#     acc = 0.0
#     count = 0
#     with torch.no_grad():
#         net.eval()
#         for img, label, meta_features in dataset:
#             label = label.to(device)
#             img = img.to(device)
#             meta_features = meta_features.to(device)
            
#             if only_img:
#                 output = model(img).squeeze()
#             elif only_meta:
#                 output = model(meta_features).squeeze()
#             else:
#                 output = model(img, meta_features).squeeze()
            
#             prediction = torch.round(output)
#             acc += torch.sum(torch.eq(prediction, label))
#             count += len(label)
#     return acc / count
            
# print(f'Train accuracy = {float(calc_accuracy(net, dataloader_train))}')
# print(f'Test accuracy = {float(calc_accuracy(net, dataloader_test))}')

## Exercise 1

The above model was okay, but not fantastic. Your job is to make a good CNN model. 

You should plot both the average loss and accuracy on both training and validation data as you train.
Finally, evaluate your model on the test data.

#### Notes to exercise 1

Feel free to take inspiration from online examples. Some nice functions to use could be: `nn.Conv2d, nn.BatchNorm2d, nn.MaxPool2d, nn.Dropout...`.

If you use `BatchNorm2d` and `Dropout`, you need to switch your network between train and evaluation mode appropriately: `net.train()`, `net.eval()`.

In the current model, we use `nn.Sigmoid` at the end. This is a bit unstable, and it is smarter to simply omit this step and use `nn.BCEWithLogitsLoss()` instead of `nn.BCELoss()`.

If you really want your model to perform, you can also use a pretrained model, e.g. `torchvision.models.resnet18`, as a backbone to your model.

## Exercise 2

Make a model that uses the meta features instead of the images (possibly using PyTorch):

In [ ]:
for img, label, meta_features in dataloader_train:
    print(meta_features[0])
    break

## Exercise 3

Combine your previous two models into one that uses both image and meta data. To do this, you should make a `Net(nn.Module)` class as demonstrated above. The `forward` function should then take both an image and a feature vector.

## Learning points:

This is a more involved CNN exercise than on the MNIST data, yet many of the learning points are the same:
1. CNNs are the goto model for image analysis.
2. They work by convoluting the input images with kernels that are trained to recognise certain features in the image (not unlike neurons in an ordinary NN).
2. Your CNN considerations should include:
     - Sample and image sizes (enough training and testing data?),
     - CNN architecture (size and number of kernels),
     - Batch size (optimising how fast you converge), and
     - if you need GPUs for the problem!
3. CNNs can be implemented in (Keras) TensorFlow (easiest) and PyTorch (harder but more versatile).

One consideration that much can be learned from is asking yourself (and peers), if you could go through the above exercise using Keras TensorFlow? If this is the case, then you can at least claim good understanding of the CNN ingredients :-)

***

## Exercise 1

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

In [ ]:
import torch.nn as nn
import torchvision.models as models
from torchvision.models import ResNet18_Weights

class HybridResNet18(nn.Module):
    def __init__(self, meta_input_dim, meta_hidden_dim=64):
        super().__init__()
        resnet = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        self.resnet_features = nn.Sequential(*list(resnet.children())[:-1])  # Remove final FC layer

        self.meta_net = nn.Sequential(
            nn.Linear(meta_input_dim, meta_hidden_dim),
            nn.ReLU(),
            nn.Linear(meta_hidden_dim, meta_hidden_dim),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.Linear(512 + meta_hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    
    def forward(self, x_img, x_meta):
        if x_img.shape[1] == 1:
            x_img = x_img.repeat(1, 3, 1, 1)  # Convert [B, 1, H, W] → [B, 3, H, W]
        x_img = self.resnet_features(x_img)
        x_img = torch.flatten(x_img, 1)
        x_meta = self.meta_net(x_meta)
        x = torch.cat((x_img, x_meta), dim=1)
        return self.classifier(x).squeeze(1)
    
    # def forward(self, x_img, x_meta):
    #     x_img = self.resnet_features(x_img)  # Pass the image through ResNet18
    #     x_img = torch.flatten(x_img, 1)      # Flatten the output
    #     x_meta = self.meta_net(x_meta)       # Pass metadata through its own network

    #     # Combine image features with metadata
    #     combined = torch.cat((x_img, x_meta), dim=1)
    #     out = self.classifier(combined)       # Final classifier layer
    #     return out



In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=10):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for x_img, y, x_meta in train_loader:
            x_img, x_meta, y = x_img.to(device), x_meta.to(device), y.float().to(device)
            optimizer.zero_grad()
            outputs = model(x_img, x_meta)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = torch.sigmoid(outputs) > 0.5
            correct += (preds == y.bool()).sum().item()
            total += y.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for x_img, y, x_meta in val_loader:
                x_img, x_meta, y = x_img.to(device), x_meta.to(device), y.float().to(device)
                outputs = model(x_img, x_meta)
                loss = criterion(outputs, y)
                val_loss += loss.item()
                preds = torch.sigmoid(outputs) > 0.5
                val_correct += (preds == y.bool()).sum().item()
                val_total += y.size(0)

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Acc={train_acc:.4f} | Val Loss={val_loss:.4f}, Acc={val_acc:.4f}")

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

    return history


def plot_training(history):
    epochs = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], label='Train Loss')
    plt.plot(epochs, history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss over Epochs')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], label='Train Acc')
    plt.plot(epochs, history['val_acc'], label='Validation Acc')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Accuracy over Epochs')
    plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
meta_input_dim = next(iter(dataloader_train))[2].shape[1]
model = HybridResNet18(meta_input_dim=meta_input_dim).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

history = train_model(model, dataloader_train, dataloader_val, criterion, optimizer, scheduler, device, num_epochs=20)
plot_training(history)


In [ ]:
def visualize_predictions(model, dataloader, device, num_images=20):
    model.eval()  # Set the model to evaluation mode
    images, labels, meta = next(iter(dataloader))  # Get a batch from the dataloader
    images, labels = images.to(device), labels.to(device)  # Move to device
    
    # Forward pass
    with torch.no_grad():
        outputs = model(images, meta)  # Forward the images and metadata through the model
        preds = torch.sigmoid(outputs).round()  # Convert logits to binary predictions
    
    # Separate the images by class
    class_0_indices = (labels == 0).nonzero(as_tuple=True)[0]  # Indices of class 0 images
    class_1_indices = (labels == 1).nonzero(as_tuple=True)[0]  # Indices of class 1 images
    
    # Select up to num_images // 2 from each class (adjust if fewer are available)
    class_0_images = class_0_indices[:num_images // 2]
    class_1_images = class_1_indices[:num_images // 2]

    # Plotting
    fig, axes = plt.subplots(2, num_images // 2, figsize=(16, 8))  # 2 rows, half num_images in each row
    axes = axes.flatten()

    # Display images of class 0 in the first row
    for i, idx in enumerate(class_0_images):
        ax = axes[i]
        ax.imshow(images[idx].cpu().permute(1, 2, 0))  # Convert to HxWxC format for RGB images
        ax.set_title(f"True: 0, Pred: {int(preds[idx].item())}")
        ax.axis('off')
    
    # Display images of class 1 in the second row
    for i, idx in enumerate(class_1_images):
        ax = axes[len(class_0_images) + i]
        ax.imshow(images[idx].cpu().permute(1, 2, 0))  # Convert to HxWxC format for RGB images
        ax.set_title(f"True: 1, Pred: {int(preds[idx].item())}")
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Example usage
visualize_predictions(model, dataloader_val, device)
